# Análisis del conteo vehicular: página web vs. conteo manual
Proyecto final de Inteligencia Artificial Aplicada (ULima, 2026-2). Código elaborado con apoyo de Claude (Anthropic, 2026).

**Qué necesitas subir a Colab (ícono de carpeta, a la izquierda):**
1. Los archivos `por_minuto_*.csv` que descarga la página, uno por video.
2. `conteo_manual.csv`, con el conteo hecho a mano por minuto (usa la plantilla).
3. Los archivos `muestra_*.csv` que descarga la página (botón "Descargar muestra"), uno por video.

Luego ejecuta las celdas en orden (Entorno de ejecución > Ejecutar todas).

In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

CLASES = ['auto', 'moto', 'bus', 'camion']
ORDEN_HORARIOS = None   # ej.: ['mañana', 'tarde', 'noche'] para fijar el orden de los gráficos

## 1. Cargar los datos (semana 5: Pandas)

In [ ]:
archivos = sorted(glob.glob('por_minuto_*.csv'))
print('Archivos de la página:', archivos)
pagina = pd.concat([pd.read_csv(a) for a in archivos], ignore_index=True)
manual = pd.read_csv('conteo_manual.csv')

for df in (pagina, manual):
    df['horario'] = df['horario'].str.strip().str.lower()

datos = manual.merge(pagina, on=['horario', 'minuto'], suffixes=('_manual', '_pagina'))
datos['total_manual'] = datos[[c + '_manual' for c in CLASES]].sum(axis=1)
datos['total_pagina'] = datos[[c + '_pagina' for c in CLASES]].sum(axis=1)
print(f'{len(datos)} minutos emparejados')
datos.head()

## 2. Análisis exploratorio (semana 6: EDA)

In [ ]:
print('Estadística descriptiva del conteo manual por minuto:')
display(manual[CLASES].describe().round(2))

composicion = manual.groupby('horario')[CLASES].sum()
composicion_pct = (composicion.div(composicion.sum(axis=1), axis=0) * 100).round(1)
if ORDEN_HORARIOS: composicion_pct = composicion_pct.loc[ORDEN_HORARIOS]
print('Composición del tráfico (% por tipo, conteo manual):')
display(composicion_pct)

veh_hora = (manual.groupby('horario')[CLASES].sum().sum(axis=1) / manual.groupby('horario')['minuto'].nunique() * 60).round(0)
print('Vehículos por hora estimados:'); display(veh_hora)

composicion_pct.plot(kind='bar', stacked=True, figsize=(7,4))
plt.ylabel('% de vehículos'); plt.title('Composición del tráfico por horario'); plt.xticks(rotation=0)
plt.legend(title='Tipo'); plt.tight_layout(); plt.savefig('fig_composicion.png', dpi=200); plt.show()

# Valores atípicos: minutos con muy pocos vehículos (semáforo en rojo, interrupciones)
z = (datos['total_manual'] - datos['total_manual'].mean()) / datos['total_manual'].std()
print('Minutos atípicos (|z| > 2):'); display(datos.loc[z.abs() > 2, ['horario', 'minuto', 'total_manual']])

## 3. Error del conteo automático por tipo y horario

In [ ]:
filas = []
for h, g in datos.groupby('horario'):
    for c in CLASES + ['total']:
        m = g[f'{c}_manual'].sum() if c != 'total' else g['total_manual'].sum()
        p = g[f'{c}_pagina'].sum() if c != 'total' else g['total_pagina'].sum()
        err = abs(p - m) / m * 100 if m > 0 else None
        filas.append({'horario': h, 'tipo': c, 'manual': m, 'pagina': p,
                      'diferencia': p - m, 'error_%': None if err is None else round(err, 1)})
errores = pd.DataFrame(filas)
display(errores)
errores.to_csv('tabla_errores.csv', index=False)

piv = errores[errores['tipo'] != 'total'].pivot(index='tipo', columns='horario', values='error_%').reindex(CLASES)
if ORDEN_HORARIOS: piv = piv[ORDEN_HORARIOS]
piv.plot(kind='bar', figsize=(7,4))
plt.ylabel('Error (%)'); plt.title('Error del conteo automático frente al manual'); plt.xticks(rotation=0)
plt.legend(title='Horario'); plt.tight_layout(); plt.savefig('fig_error.png', dpi=200); plt.show()

## 4. Matriz de confusión con la muestra de 50 vehículos (semana 7: métricas)

In [ ]:
arch_m = sorted(glob.glob('muestra_*.csv'))
print('Archivos de muestra:', arch_m)
muestra = pd.concat([pd.read_csv(a) for a in arch_m], ignore_index=True)
print(f'{len(muestra)} vehículos en la muestra')
muestra['clase_real'] = muestra['clase_real'].str.strip().str.lower()
muestra['clase_pagina'] = muestra['clase_pagina'].str.strip().str.lower()

# Tabla completa: incluye combi y cúster, que COCO-SSD no conoce
print('Con qué clase registró la página cada tipo real:')
display(pd.crosstab(muestra['clase_real'], muestra['clase_pagina'], margins=True))

# Métricas solo con las 4 clases que el modelo sí conoce
conocidas = muestra[muestra['clase_real'].isin(CLASES)]
etiquetas = [c for c in CLASES if c in set(conocidas['clase_real']) | set(conocidas['clase_pagina'])]
print(classification_report(conocidas['clase_real'], conocidas['clase_pagina'], labels=etiquetas, zero_division=0))
# Figura: todas las clases reales (incluye combi, cúster y "ninguno") contra lo que dijo la página
orden_real = [c for c in ['auto','moto','bus','combi','custer','camion','mototaxi','ninguno'] if c in set(muestra['clase_real'])]
tabla = pd.crosstab(muestra['clase_real'], muestra['clase_pagina']).reindex(index=orden_real, columns=CLASES, fill_value=0)
fig, ax = plt.subplots(figsize=(6, 4.6))
im = ax.imshow(tabla.values, cmap='Oranges')
ax.set_xticks(range(len(CLASES)), CLASES); ax.set_yticks(range(len(orden_real)), orden_real)
for i in range(tabla.shape[0]):
    for j in range(tabla.shape[1]):
        v = tabla.values[i, j]
        ax.text(j, i, v, ha='center', va='center', color='white' if v > tabla.values.max()/2 else 'black')
ax.set_xlabel('Clase asignada por la página'); ax.set_ylabel('Clase real')
ax.set_title(f'Clasificación de la muestra ({len(muestra)} vehículos)')
plt.tight_layout(); plt.savefig('fig_matriz_confusion.png', dpi=200); plt.show()

## 5. Regresión manual vs. página (semana 8)
Se exporta `datos_regresion.csv` para hacer la regresión en **Orange**: File → Linear Regression → Test and Score (target: `total_manual`). La celda también la calcula en Python para comparar.

In [ ]:
reg = datos[['horario', 'minuto', 'total_pagina', 'total_manual']]
reg.to_csv('datos_regresion.csv', index=False)

X, y = reg[['total_pagina']], reg['total_manual']
lr = LinearRegression().fit(X, y)
pred = lr.predict(X)
print(f'total_manual = {lr.coef_[0]:.3f} x total_pagina + {lr.intercept_:.2f}')
print(f'R2 = {r2_score(y, pred):.3f}   MAE = {mean_absolute_error(y, pred):.2f} vehículos por minuto')

plt.figure(figsize=(5,5))
for h in (ORDEN_HORARIOS or sorted(reg['horario'].unique())):
    g = reg[reg['horario'] == h]
    plt.scatter(g['total_pagina'], g['total_manual'], label=h)
lim = [0, max(reg['total_pagina'].max(), reg['total_manual'].max()) + 2]
plt.plot(lim, lim, '--', color='gray', label='Conteo perfecto')
plt.plot(lim, lr.predict(pd.DataFrame({'total_pagina': lim})), color='black', label='Regresión')
plt.xlabel('Vehículos por minuto (página)'); plt.ylabel('Vehículos por minuto (manual)')
plt.legend(); plt.title('Conteo manual vs. página por minuto'); plt.tight_layout()
plt.savefig('fig_regresion.png', dpi=200); plt.show()

## 6. (Opcional) Efecto del umbral de confianza
Pasa un mismo video por la página con umbrales 0.3, 0.5 y 0.7. Cambia el campo *Horario* en la página a `umbral_0.3`, `umbral_0.5`, etc. y sube esos CSV con el nombre `umbral_*.csv`.

In [ ]:
arch_u = sorted(glob.glob('umbral_*.csv'))
if arch_u:
    u = pd.concat([pd.read_csv(a) for a in arch_u])
    tabla_u = u.groupby('horario')[CLASES + ['total']].sum()
    display(tabla_u)
    tabla_u['total'].plot(kind='bar', figsize=(6,4), color='#E8741E')
    plt.ylabel('Vehículos contados'); plt.title('Conteo según el umbral de confianza'); plt.xticks(rotation=0)
    plt.tight_layout(); plt.savefig('fig_umbral.png', dpi=200); plt.show()
else:
    print('No se subieron archivos umbral_*.csv; se omite esta sección.')